In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
# NOTE: Hardcoded for portfolio/dev environment.
# In production, this would be passed via:
    # - Databricks Widgets for interactive runs
    # - Job parameters for scheduled pipeline runs
    # - Databricks Secrets for sensitive credentials
CATALOG = "olist_ecommerce"


print("Silver layer setup complete!")

In [0]:
def dq_check(df, table_name, critical_rules, warn_rules=[]):
    """
    Data Quality checks:
    - critical_rules: DROP rows vi phạm
    - warn_rules: LOG nhưng không drop
    """
    initial = df.count()
    result  = df

    # Critical: drop
    for col_name, condition in critical_rules:
        before = result.count()
        result = result.filter(condition)
        dropped = before - result.count()
        if dropped > 0:
            print(f"[CRITICAL] {table_name}.{col_name}: dropped {dropped:,} rows")

    # Warn: log only
    for col_name, condition in warn_rules:
        violations = result.filter(f"NOT ({condition})").count()
        if violations > 0:
            print(f"[WARN] {table_name}.{col_name}: {violations:,} rows vi phạm")

    final = result.count()
    print(f"[DQ OK] {table_name}: {initial:,} → {final:,} rows ({initial - final:,} dropped)\n")
    return result